# Panel dataset construction (superseded — kept for the record)

Builds the point-in-time **panel**: one row per (company, cutoff T), label = took a first Lloyds
charge in (T, T+13 weeks].

**This design was superseded.** At 0.02% row-level prevalence (426 positives in 2.1M rows) the
temporal panel could not lift Precision@K off zero, and the media ablation on it showed
`tone_z`/`vol_z` contribute nothing (clustered CIs cross zero). The project moved to the flat
look-alike design in `data_flat.ipynb` and the matched case-control design in
`data_conditional.ipynb`.

Kept because it is the evidence behind that decision — the negative result is a finding, and the
methodology section of the report depends on it.

In [2]:
# ===========================================================================
# Part 6 — build_panel(): one row per (company, cutoff T), features + forward label
# ===========================================================================
import os
import pandas as pd
import numpy as np

# --- SIC -> the 21 Companies House SIC SECTIONS (A-U) -----------------------
# Same taxonomy as the model notebook, so EVERY company gets a section AND the
# labels match the GDELT media_index (Part 3 SECTOR_PATTERNS). This is what fixes
# most media_missing (previously ~24% of firms had no sector at all).
SIC_SECTIONS = [
    (1, 3,   "A: Agriculture"),           (5, 9,   "B: Mining"),
    (10, 33, "C: Manufacturing"),         (35, 35, "D: Utilities"),
    (36, 39, "E: Water/Waste"),           (41, 43, "F: Construction"),
    (45, 47, "G: Retail/Wholesale"),      (49, 53, "H: Transport"),
    (55, 56, "I: Accommodation/Food"),    (58, 63, "J: Information/Comms"),
    (64, 66, "K: Finance/Insurance"),     (68, 68, "L: Real Estate"),
    (69, 75, "M: Professional/Scientific"), (77, 82, "N: Admin Support"),
    (84, 84, "O: Public Admin"),          (85, 85, "P: Education"),
    (86, 88, "Q: Health/Social"),         (90, 93, "R: Arts/Recreation"),
    (94, 96, "S: Other Services"),        (97, 98, "T: Household Activities"),
    (99, 99, "U: Extraterritorial"),
]
# SIC 98 (resident property-management cos.) is officially section T (Household),
# which has ~no funding news -> media_missing. Override it to Real Estate so those
# ~4k firms keep a media signal. Empty this dict for a pure SIC_SECTIONS mapping.
SECTION_OVERRIDE = {98: "L: Real Estate"}


def code_to_section(code):
    """Map a SIC code to its Companies House SIC section label (A-U), or None."""
    try:
        div = int(str(code).strip()[:2])
    except (ValueError, TypeError):
        return None
    if div in SECTION_OVERRIDE:
        return SECTION_OVERRIDE[div]
    for lo, hi, section in SIC_SECTIONS:
        if lo <= div <= hi:
            return section
    return None


def _floor_to_week(ts):
    """Floor a date to the Sunday-starting week (matches BigQuery DATE_TRUNC WEEK)."""
    ts = pd.Timestamp(ts)
    return ts - pd.Timedelta(days=(ts.dayofweek + 1) % 7)


def build_panel(companies, charges, media_index, cutoffs,
                horizon_weeks=13, exclude_dissolved=False):
    """One row per (company, cutoff T): point-in-time features + forward label.

    label = 1 if the firm registered its FIRST Lloyds-group charge in (T, T+horizon].
    Excluded at T if it already held a Lloyds charge on/before T (existing customer)
    or did not yet exist at T. Media tone_z/vol_z are joined at week(T).

    companies: SME table (com_num, sic_code, date_of_creation, region, ...)
    charges:   charges_history.csv (com_num, created_on, is_lloyds)
    media_index: Part 4 index (sector, region, week, tone_z, vol_z, z_source)
    cutoffs:   list of cutoff dates T (strings or Timestamps)
    """
    comp = companies.copy()
    comp["date_of_creation"] = pd.to_datetime(comp["date_of_creation"], errors="coerce")
    comp["sector"] = comp["sic_code"].map(code_to_section)

    ch = charges.copy()
    ch["created_on"] = pd.to_datetime(ch["created_on"], errors="coerce")
    ch["is_lloyds"]  = ch["is_lloyds"].astype(str).str.lower().eq("true")
    ch = ch.dropna(subset=["created_on"])
    first_lloyds = ch[ch["is_lloyds"]].groupby("com_num")["created_on"].min()   # per firm
    nonlloyds    = ch[~ch["is_lloyds"]]

    mi = media_index.copy()
    mi["week"] = pd.to_datetime(mi["week"])
    media_weeks = np.sort(mi["week"].unique())      # for snap-to-nearest-prior-week

    frames = []
    for T in pd.to_datetime(list(cutoffs)):
        wk = _floor_to_week(T)
        horizon_end = T + pd.Timedelta(weeks=horizon_weeks)

        # eligible = existed by T AND no Lloyds charge on/before T
        fll = comp["com_num"].map(first_lloyds)
        eligible = (fll.isna() | (fll > T)) & comp["date_of_creation"].le(T)
        if exclude_dissolved and "company_status" in comp.columns:
            eligible &= comp["company_status"].eq("active")
        sub = comp[eligible].copy()

        # forward label: first Lloyds charge lands in (T, T+horizon]
        fll_sub = sub["com_num"].map(first_lloyds)
        sub["label"] = ((fll_sub > T) & (fll_sub <= horizon_end)).astype(int)

        # point-in-time features
        sub["age_years"] = (T - sub["date_of_creation"]).dt.days / 365.25
        pnl = nonlloyds[nonlloyds["created_on"] <= T].groupby("com_num").size()
        sub["prior_nonlloyds_charges"] = sub["com_num"].map(pnl).fillna(0).astype(int)
        sub["T"], sub["week"] = T, wk

        # media join: snap to the most recent media week AT OR BEFORE week(T)
        # (point-in-time safe; tolerates a cutoff off the index's exact week grid)
        _prior = media_weeks[media_weeks <= wk]
        use_wk = _prior.max() if len(_prior) else pd.NaT
        mslice = mi.loc[mi["week"] == use_wk, ["sector", "region", "tone_z", "vol_z", "z_source"]]
        sub = sub.merge(mslice, on=["sector", "region"], how="left")
        sub["media_missing"] = sub["tone_z"].isna().astype(int)
        frames.append(sub)

    keep = ["com_num", "name", "T", "week", "age_years", "sector", "region",
            "account_type", "accounts_overdue", "prior_nonlloyds_charges",
            "tone_z", "vol_z", "z_source", "media_missing", "label"]
    panel = pd.concat(frames, ignore_index=True)
    return panel[[c for c in keep if c in panel.columns]]


# --- run it ----------------------------------------------------------------
# --- portable paths: resolve the project root from ANY working directory ---
import sys
from pathlib import Path
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "API").is_dir())
sys.path.insert(0, str(_ROOT))
from paths import SME_CSV, CHARGES_CSV, PANEL_CSV

PANEL_OUT = str(PANEL_CSV)
_cache_dir  = next((d for d in ("GDELT/BigQuery Cache files", "BigQuery Cache files")
                    if os.path.isdir(d)), "GDELT/BigQuery Cache files")
MEDIA_INDEX = os.path.join(_cache_dir, "context_media_index_104w.parquet")

companies   = pd.read_csv(SME_CSV, dtype=str)
charges     = pd.read_csv(CHARGES_CSV, dtype=str)
media_index = pd.read_parquet(MEDIA_INDEX)

# Cutoffs must sit inside the media window with room for the horizon after them.
# Non-overlapping = spaced by the horizon so each conversion counts once.
HORIZON_WEEKS = 13   # = cutoff spacing, so label windows tile exactly (no overlap)
CUTOFFS = ["2024-10-13", "2025-01-12", "2025-04-13", "2025-07-13",   # 7 quarterly cutoffs,
           "2025-10-12", "2026-01-11", "2026-04-12"]   # all real index weeks in the 104w window

panel = build_panel(companies, charges, media_index, CUTOFFS,
                    horizon_weeks=HORIZON_WEEKS)
panel.to_csv(PANEL_OUT, index=False)

# --- report ----------------------------------------------------------------
print(f"panel: {len(panel)} rows ({panel['com_num'].nunique()} companies x "
      f"{len(CUTOFFS)} cutoffs)  ->  '{PANEL_OUT}'")
print("\nper-cutoff rows / positives:")
print(panel.groupby("T").agg(rows=("com_num", "size"),
                             positives=("label", "sum")).to_string())
print(f"\ntotal positives: {int(panel['label'].sum())}  "
      f"(prevalence {panel['label'].mean():.2%})")
print(f"media features filled: {(panel['media_missing'] == 0).mean():.0%}  |  "
      f"sector mapped: {panel['sector'].notna().mean():.0%}")
if (panel["media_missing"] == 0).mean() < 0.20:
    print("\n⚠️  Almost no media matched — the media_index sector labels likely "
          "predate the SIC-section taxonomy. Re-pull Parts 3 & 4 to rebuild it.")
print("\nsample:")
print(panel.head(4).to_string(index=False))

panel: 2117863 rows (303919 companies x 7 cutoffs)  ->  '/Users/natchalin_/Desktop/final_project/Lloyds/API/panel.csv'

per-cutoff rows / positives:
              rows  positives
T                            
2024-10-13  298636         35
2025-01-12  301551         62
2025-04-13  303068         36
2025-07-13  303652         40
2025-10-12  303710         73
2026-01-11  303665         92
2026-04-12  303581         88

total positives: 426  (prevalence 0.02%)
media features filled: 89%  |  sector mapped: 100%

sample:
 com_num                       name          T       week  age_years                     sector region         account_type accounts_overdue  prior_nonlloyds_charges  tone_z  vol_z z_source  media_missing  label
15073164 NFLECTION ADVISORY LIMITED 2024-10-13 2024-10-13   1.163587 M: Professional/Scientific    NaN total-exemption-full            False                        0     NaN   <NA>      NaN              1      0
13522064               NFOGENIE LTD 2024-10-13 2024-10-

## Point-in-time filing features (joined onto the panel)

In [ ]:
# ===========================================================================
# Part 6c — point-in-time FILING features (leakage-safe) joined onto the panel
# ===========================================================================
# Due date approximated as made_up_date + 274 days (~9 months, private-co deadline).
# All features use only filings with date <= T (point-in-time). Missing -> impute +
# a filings_missing flag (same policy as media). Refit imputation on TRAIN in the model.
# --- portable paths: resolve the project root from ANY working directory ---
import sys
from pathlib import Path
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "API").is_dir())
sys.path.insert(0, str(_ROOT))
from paths import FILINGS_CSV

if not os.path.exists(FILINGS_CSV):
    print(f"filings_history.csv not found -> run Stage 6 of the CH pipeline first.\n  {FILINGS_CSV}")
else:
    fil = pd.read_csv(FILINGS_CSV, dtype=str)
    fa = fil[fil["category"] == "accounts"].copy()
    fa["date"] = pd.to_datetime(fa["date"], errors="coerce")
    fa["made_up_date"] = pd.to_datetime(fa["made_up_date"], errors="coerce")
    fa = fa.dropna(subset=["date"])
    fa["due"] = fa["made_up_date"] + pd.Timedelta(days=274)     # ~9 months
    fa["days_late"] = (fa["date"] - fa["due"]).dt.days
    fa["late"] = fa["days_late"] > 0

    panel = pd.read_csv(PANEL_OUT, dtype={"com_num": str})      # PANEL_OUT from Part 6
    panel["_Tdt"] = pd.to_datetime(panel["T"])                  # keep original T string intact

    feat_frames = []
    for T in pd.to_datetime(panel["_Tdt"].unique()):
        sub = fa[fa["date"] <= T]                               # filed on/before T
        g = sub.groupby("com_num")
        mu = sub.loc[sub["made_up_date"] <= T].groupby("com_num")["made_up_date"].max()
        f = pd.DataFrame({
            "n_accounts_filed": g.size(),
            "pct_late":         g["late"].mean(),
            "avg_days_late":    g["days_late"].clip(lower=0).mean(),
        })
        f["accounts_staleness_months"] = (T - mu).dt.days / 30.44
        f["_Tdt"] = T
        feat_frames.append(f.reset_index())
    feats = pd.concat(feat_frames, ignore_index=True)

    new_cols = ["n_accounts_filed", "pct_late", "avg_days_late",
                "accounts_staleness_months", "filings_missing"]
    panel = panel.drop(columns=[c for c in new_cols if c in panel.columns], errors="ignore")
    panel = panel.merge(feats, on=["com_num", "_Tdt"], how="left").drop(columns="_Tdt")

    panel["filings_missing"] = panel["n_accounts_filed"].isna().astype(int)
    panel["n_accounts_filed"] = panel["n_accounts_filed"].fillna(0).astype(int)
    panel["pct_late"] = panel["pct_late"].fillna(0.0)
    panel["avg_days_late"] = panel["avg_days_late"].fillna(0.0)
    panel["accounts_staleness_months"] = panel["accounts_staleness_months"].fillna(
        panel["accounts_staleness_months"].median())

    panel.to_csv(PANEL_OUT, index=False)
    print(f"Added filing features -> {PANEL_OUT}")
    print(panel[new_cols].describe().round(2).to_string())
    print(f"\nfilings_missing: {panel['filings_missing'].mean():.1%} of rows "
          f"({panel.loc[panel['filings_missing']==0,'com_num'].nunique()} companies have accounts)")